# Agentic AI for Drug Safety & Discovery
### LangChain · local Ollama LLM via `ChatOllama` (`nemotron-3.5-lightning:30b-mlx`) · real FDA + PubMed data

A multi-agent system that answers two questions pharma teams ask every day, and connects them:

1. **Drug Safety:** *"Is drug X linked to adverse event Y?"*
   Disproportionality statistics on the FDA Adverse Event Reporting System (FAERS) + an LLM clinical review of
   PubMed, checked by a **quality-control agent that sends targeted feedback** until the analysis passes.
2. **Drug Discovery:** *"Which existing drugs could be repurposed for disease Z?"*
   **Agentic RAG** over PubMed: the agent grades its own retrieval, **rewrites its search when the evidence is
   weak**, extracts candidates as validated structured data and ranks them with a transparent score.
3. **Safety-aware discovery:** every repurposing candidate is **screened against FAERS** before it is recommended.

```
                       ┌──────────────────────── Part A: DRUG SAFETY ─────────────────────────┐
 drug + event ───────▶ │ FAERS 2×2 → Statistician ─▶ Clinical reviewer ─▶ QC agent ─┐          │──▶ safety report
                       │                   ▲                ▲                         │ feedback │
                       │                   └──── targeted re-runs ◀──────────────────┘          │
                       └───────────────────────────────────────────────────────────────────────┘
                       ┌────────────────────── Part B: DRUG DISCOVERY ────────────────────────┐
 disease ────────────▶ │ PubMed → embed → retrieve → self-grade ──low──▶ Query rewriter ─┐     │
                       │                       │ ok                    (new searches) ◀──┘     │
                       │                       ▼                                              │
                       │            Evidence extractor → PMID guardrail → scoring             │
                       └───────────────────────────────┬───────────────────────────────────────┘
                                                       ▼
                        Part C: FAERS safety screen of each candidate ─▶ cited brief (markdown)
```

**Engineering highlights**
* **Runs fully locally**: the LLM and the embeddings run on Ollama; no patient or company data leaves the machine.
* **Structured outputs everywhere**: every LLM answer is a validated Pydantic object, never free-form JSON.
* **Deterministic where it matters**: statistics, QC rules, scores and risk levels are plain Python, so they are
  reproducible and auditable; the LLM is used for judgement and language.
* **Guardrails**: cited PMIDs must exist in the retrieved evidence; evidence strength must match the literature count.
* **Audit trail**: every agent action is appended to `runs/<run_id>/audit.jsonl`.
* **Two implementations**: this notebook (LangChain) and a twin built with Pydantic AI, with identical logic.

> ⚠️ Research/educational project. Not for clinical or regulatory decisions.

## 1. Setup
```bash
ollama pull nemotron-3.5-lightning:30b-mlx       # chat model
ollama pull nomic-embed-text                     # embedding model
ollama serve                                     # if the Ollama app is not already running
```
Optional `.env`: `ENTREZ_EMAIL=you@example.com` (NCBI etiquette), `NCBI_API_KEY=...` (higher PubMed rate limit).

In [ ]:
# %pip install -q langchain-core langchain-ollama requests numpy pandas python-dotenv

In [ ]:
import hashlib, json, math, os, re, time, uuid
import xml.etree.ElementTree as ET
from datetime import date, datetime
from typing import Any, Dict, List, Literal, Optional

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

MODEL_NAME = "nemotron-3.5-lightning:30b-mlx"
EMBED_MODEL = "nomic-embed-text"
ENTREZ_EMAIL = os.getenv("ENTREZ_EMAIL", "student@example.com")
NCBI_API_KEY = os.getenv("NCBI_API_KEY")

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:4]
RUN_DIR = os.path.join("runs", RUN_ID)
os.makedirs(RUN_DIR, exist_ok=True)
print("Run:", RUN_ID, "| model:", MODEL_NAME, "| embeddings:", EMBED_MODEL)

## 2. Shared infrastructure (no LLM)
### 2.1 Cached HTTP + audit log
API responses are cached on disk, so re-running the notebook is fast and reproducible.

In [ ]:
CACHE_DIR = "cache"
os.makedirs(CACHE_DIR, exist_ok=True)


def http_get(url: str, params: Dict[str, Any], as_json: bool = True, pause: float = 0.0):
    """GET with a small disk cache. Returns parsed JSON (or text), or None on a 404."""
    key = hashlib.sha256(json.dumps([url, params], sort_keys=True).encode()).hexdigest()[:32]
    path = os.path.join(CACHE_DIR, key + (".json" if as_json else ".txt"))
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f) if as_json else f.read()
    time.sleep(pause)
    r = requests.get(url, params=params, timeout=30)
    if r.status_code == 404:
        return None
    r.raise_for_status()
    data = r.json() if as_json else r.text
    with open(path, "w") as f:
        json.dump(data, f) if as_json else f.write(data)
    return data


class AuditLog:
    """Append-only JSONL audit trail (one line per agent action)."""
    def __init__(self, path: str):
        self.path = path

    def log(self, agent: str, action: str, **data):
        entry = {"ts": datetime.now().isoformat(timespec="seconds"), "agent": agent, "action": action, **data}
        with open(self.path, "a") as f:
            f.write(json.dumps(entry, default=str) + "\n")


audit = AuditLog(os.path.join(RUN_DIR, "audit.jsonl"))

### 2.2 PubMed client (NCBI E-utilities)

In [ ]:
EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"


def _ncbi(params):
    p = {"tool": "drug-safety-discovery", "email": ENTREZ_EMAIL, **params}
    if NCBI_API_KEY:
        p["api_key"] = NCBI_API_KEY
    return p


def pubmed_search(query: str, retmax: int = 20) -> List[Dict[str, Any]]:
    """Search PubMed and return articles: pmid, title, abstract, year, mesh."""
    res = http_get(f"{EUTILS}/esearch.fcgi", _ncbi({"db": "pubmed", "term": query, "retmax": retmax,
                                                    "retmode": "json", "sort": "relevance"}), pause=0.34)
    ids = (res or {}).get("esearchresult", {}).get("idlist", [])
    if not ids:
        return []
    xml = http_get(f"{EUTILS}/efetch.fcgi", _ncbi({"db": "pubmed", "id": ",".join(ids), "retmode": "xml"}),
                   as_json=False, pause=0.34)
    articles = []
    for art in ET.fromstring(xml).findall(".//PubmedArticle"):
        title_el = art.find(".//ArticleTitle")
        year = art.findtext(".//JournalIssue/PubDate/Year") or (art.findtext(".//JournalIssue/PubDate/MedlineDate") or "")[:4]
        articles.append({
            "pmid": art.findtext(".//MedlineCitation/PMID"),
            "title": "".join(title_el.itertext()) if title_el is not None else "",
            "abstract": " ".join("".join(t.itertext()) for t in art.findall(".//Abstract/AbstractText")),
            "year": int(year) if year.isdigit() else None,
            "mesh": [d.text for d in art.findall(".//MeshHeading/DescriptorName")],
        })
    return articles

### 2.3 FAERS client (openFDA) + disproportionality statistics

For a drug–event pair we build the classic 2×2 table over all FAERS reports:

| | event | other events |
|---|---|---|
| **drug** | a | b |
| **other drugs** | c | d |

* **ROR** (reporting odds ratio) = (a/b)/(c/d), with a 95 % CI on the log scale
* **PRR** (proportional reporting ratio) = [a/(a+b)] / [c/(c+d)]
* **Signal** (EMA-style): a ≥ 3 **and** lower 95 % CI of ROR > 1

In [ ]:
OPENFDA = "https://api.fda.gov/drug/event.json"


def faers_total(search: Optional[str] = None) -> int:
    params = {"limit": 1, **({"search": search} if search else {})}
    res = http_get(OPENFDA, params, pause=0.25)
    return int((res or {}).get("meta", {}).get("results", {}).get("total", 0))


def drug_q(drug: str) -> str:
    return f'patient.drug.medicinalproduct:"{drug.upper()}"'


def event_q(event: str) -> str:
    return f'patient.reaction.reactionmeddrapt:"{event.upper()}"'


def disproportionality(drug: str, event: str) -> Dict[str, Any]:
    a = faers_total(f"{drug_q(drug)} AND {event_q(event)}")
    n_drug, n_event, n_all = faers_total(drug_q(drug)), faers_total(event_q(event)), faers_total()
    b, c = max(n_drug - a, 0), max(n_event - a, 0)
    d = max(n_all - a - b - c, 0)
    aa, bb, cc, dd = (x + 0.5 for x in (a, b, c, d)) if 0 in (a, b, c, d) else (a, b, c, d)   # Haldane correction
    ror = (aa / bb) / (cc / dd)
    se = math.sqrt(1 / aa + 1 / bb + 1 / cc + 1 / dd)
    lo, hi = math.exp(math.log(ror) - 1.96 * se), math.exp(math.log(ror) + 1.96 * se)
    prr = (aa / (aa + bb)) / (cc / (cc + dd))
    # chi-square (1 df) and its p-value
    n = aa + bb + cc + dd
    chi2 = n * (aa * dd - bb * cc) ** 2 / ((aa + bb) * (cc + dd) * (aa + cc) * (bb + dd))
    p = math.erfc(math.sqrt(chi2 / 2))
    return {"drug": drug.upper(), "event": event.upper(), "table": {"a": a, "b": b, "c": c, "d": d},
            "ror": round(ror, 2), "ror_ci95": [round(lo, 2), round(hi, 2)], "prr": round(prr, 2),
            "chi2": round(chi2, 1), "p_value": float(f"{p:.3g}"),
            "signal": bool(a >= 3 and lo > 1)}


def safety_snapshot(drug: str) -> Dict[str, Any]:
    """Quick FAERS profile of a drug: report volume, share of serious reports, top reactions."""
    total = faers_total(drug_q(drug))
    serious = faers_total(f"{drug_q(drug)} AND serious:1") if total else 0
    top = http_get(OPENFDA, {"search": drug_q(drug), "count": "patient.reaction.reactionmeddrapt.exact", "limit": 5},
                   pause=0.25) if total else None
    return {"drug": drug, "faers_reports": total, "serious_ratio": round(serious / total, 2) if total else None,
            "top_reactions": [r["term"] for r in (top or {}).get("results", [])]}

## 3. The LLM layer (LangChain + Ollama)
`ChatOllama` runs the local model. Every agent = **chat model + system prompt + output type**:
for Pydantic output types we use `llm.with_structured_output(schema)`, which uses Ollama's JSON-schema mode and
parses the reply into a validated object. (If the model struggles, try `method="json_mode"` in `LLMAgent`.)

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama, OllamaEmbeddings

llm = ChatOllama(
    model="nemotron-3.5-lightning:30b-mlx",
    temperature=0.1,
    num_ctx=16384,      # Ollama's default context window is small; our prompts carry abstracts / JSON
)
embeddings = OllamaEmbeddings(model=EMBED_MODEL)


class LLMAgent:
    """system prompt + chat model + output type. .invoke(prompt) returns a validated object (or text)."""
    def __init__(self, llm, output_type=str, system_prompt: str = ""):
        self.system_prompt, self.output_type = system_prompt, output_type
        self.runnable = llm if output_type is str else llm.with_structured_output(output_type)

    def invoke(self, prompt: str):
        out = self.runnable.invoke([SystemMessage(self.system_prompt), HumanMessage(prompt)])
        if self.output_type is str:      # reasoning models may wrap their thoughts in <think> tags
            return re.sub(r"<think>.*?</think>", "", out.content, flags=re.S).strip()
        return out


def ask(agent: LLMAgent, prompt: str):
    return agent.invoke(prompt)


def embed(texts: List[str]) -> np.ndarray:
    """Embed texts locally and L2-normalise them (so a dot product = cosine similarity)."""
    v = np.array(embeddings.embed_documents(texts), dtype=float)
    return v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-12)

# Part A: Drug Safety (signal detection with a QC feedback loop)

| Agent | Kind | Job |
|---|---|---|
| **Statistician** | LLM | Interprets the FAERS statistics: signal or not, key numbers, limitations |
| **Clinical reviewer** | LLM | Reads PubMed abstracts: mechanism, confounders, PLAUSIBLE / IMPLAUSIBLE / UNCERTAIN |
| **QC agent** | rules | Scores the analysis 0–100 and sends each issue to the agent that must fix it |
| **Reporter** | rules | Risk level + recommended actions (a fixed matrix, so it is auditable) |

In [ ]:
class StatReview(BaseModel):
    signal_detected: bool = Field(description="True if the data meet the signal criteria")
    key_numbers: str = Field(description="One line quoting ROR with 95% CI, PRR, case count a and p-value")
    interpretation: str = Field(description="3-5 sentences, statistical only (no clinical advice)")
    limitations: List[str] = Field(description="Data limitations, e.g. reporting bias, low counts, confounding by indication")
    feedback_response: str = Field(description="How QC feedback was addressed; empty string on the first pass")


class ClinicalReview(BaseModel):
    classification: Literal["PLAUSIBLE", "IMPLAUSIBLE", "UNCERTAIN"]
    evidence_strength: Literal["none", "limited", "moderate", "strong"] = Field(
        description="Must match the number of PubMed articles provided: 0=none, 1-4=limited, 5-14=moderate, 15+=strong")
    mechanism: str = Field(description="Most likely biological mechanism / pathway, or why it is unknown")
    confounders: List[str] = Field(description="Alternative explanations, e.g. indication, co-medication, reporting bias")
    narrative: str = Field(description="Clinical narrative: association vs causation, mechanism, confounders, uncertainty")
    cited_pmids: List[str] = Field(description="PMIDs (digits) from the provided articles that support the narrative")
    feedback_response: str = Field(description="How QC feedback was addressed; empty string on the first pass")


statistician = LLMAgent(llm, output_type=StatReview, system_prompt=(
    "You are a senior pharmacovigilance biostatistician. Interpret FAERS disproportionality statistics. "
    "A signal requires a >= 3 cases AND the lower 95% CI of the ROR > 1. Quote the exact numbers you are given. "
    "Explain limitations of spontaneous-report data. No clinical recommendations. "
    "If the message contains QC FEEDBACK, fix every point and explain how in feedback_response."))

clinical_reviewer = LLMAgent(llm, output_type=ClinicalReview, system_prompt=(
    "You are a clinical pharmacologist reviewing a possible adverse drug reaction. Use ONLY the abstracts provided. "
    "Distinguish association from causation (Bradford Hill: strength, consistency, temporality, dose-response, "
    "biological plausibility). Discuss the mechanism and confounders. Classify as PLAUSIBLE, IMPLAUSIBLE or UNCERTAIN. "
    "evidence_strength must match the article count you are told. Cite only PMIDs from the list. "
    "If the message contains QC FEEDBACK, fix every point and explain how in feedback_response."))


def strength_from_count(n: int) -> str:
    return "none" if n == 0 else "limited" if n < 5 else "moderate" if n < 15 else "strong"


def format_articles(articles: List[Dict], n: int = 8, chars: int = 700) -> str:
    return "\n\n".join(f"PMID {a['pmid']} ({a['year']}): {a['title']}\n{a['abstract'][:chars]}" for a in articles[:n])

### The QC agent: explicit, testable rules
Each rule returns an issue with a **severity**, the **agent responsible** and a **specific fix**. The score is
100 minus penalties (high = 25, medium = 10, low = 0). Because the rules are code, the same analysis always gets
the same score, and a reviewer can see exactly *why* something failed.

In [ ]:
PENALTY = {"high": 25, "medium": 10, "low": 0}
MECHANISM_WORDS = ("mechanism", "pathway", "receptor", "confound", "causal", "bradford", "dose", "temporal",
                   "plausib", "inhibit", "hypothes")


def qc_review(stats: Dict, stat: StatReview, clin: ClinicalReview, articles: List[Dict]) -> Dict[str, Any]:
    issues = []

    def issue(agent, severity, problem, fix):
        issues.append({"agent": agent, "severity": severity, "problem": problem, "fix": fix})

    # Statistician: must agree with the computed statistics and quote them
    if stat.signal_detected != stats["signal"]:
        issue("statistician", "high", f"signal_detected={stat.signal_detected} but the criteria give {stats['signal']}",
              f"Re-apply the criteria: a={stats['table']['a']}, ROR CI={stats['ror_ci95']}.")
    if str(stats["ror"]) not in stat.key_numbers:
        issue("statistician", "medium", "key_numbers does not quote the computed ROR",
              f"Quote ROR {stats['ror']} (95% CI {stats['ror_ci95'][0]}-{stats['ror_ci95'][1]}), PRR {stats['prr']}.")
    # Clinical reviewer: evidence must match the literature, citations must be real
    expected = strength_from_count(len(articles))
    if clin.evidence_strength != expected:
        issue("clinical", "high", f"evidence_strength '{clin.evidence_strength}' but {len(articles)} articles were found",
              f"Set evidence_strength to '{expected}' and adjust the narrative accordingly.")
    fake = [p for p in clin.cited_pmids if p not in {a["pmid"] for a in articles}]
    if fake:
        issue("clinical", "high", f"cited PMIDs not in the evidence: {fake}", "Cite only PMIDs from the provided list.")
    hits = sum(w in clin.narrative.lower() for w in MECHANISM_WORDS)
    if stats["signal"] and clin.classification != "PLAUSIBLE" and hits < 3:
        issue("clinical", "medium", "Statistical signal but the narrative does not investigate mechanism/confounding",
              "Propose a mechanistic hypothesis, list confounders, and state what evidence would resolve the uncertainty.")
    if stats["signal"] and clin.classification == "IMPLAUSIBLE" and not clin.confounders:
        issue("clinical", "high", "Signal judged IMPLAUSIBLE without an alternative explanation",
              "Name the confounders (e.g. indication, co-medication, stimulated reporting) that explain the signal.")
    score = max(0, 100 - sum(PENALTY[i["severity"]] for i in issues))
    return {"score": score, "issues": issues, "targets": sorted({i["agent"] for i in issues})}


def feedback_for(agent: str, issues: List[Dict]) -> str:
    return "\n".join(f"- [{i['severity'].upper()}] {i['problem']} → {i['fix']}" for i in issues if i["agent"] == agent)


RISK_ACTIONS = {
    "HIGH": ["Update product labelling", "Healthcare professional communication", "Consider risk-minimisation measures"],
    "MODERATE": ["Enhanced surveillance", "Targeted follow-up of new reports", "Re-assess at next periodic safety report"],
    "LOW": ["Routine signal monitoring"],
}


def regulatory_report(stats: Dict, clin: ClinicalReview) -> Dict[str, Any]:
    """Fixed risk matrix: statistical signal × clinical plausibility."""
    plausible = clin.classification == "PLAUSIBLE"
    risk = "HIGH" if stats["signal"] and plausible else "MODERATE" if stats["signal"] or plausible else "LOW"
    return {"signal": stats["signal"], "plausibility": clin.classification, "evidence": clin.evidence_strength,
            "risk_level": risk, "actions": RISK_ACTIONS[risk]}

### The safety pipeline with the feedback loop
Statistician → clinical reviewer → QC. While the score is below the threshold, **only the agents QC flagged are
re-run**, each with its own to-do list, until it passes, nothing is flagged, or the iteration budget runs out.

In [ ]:
def run_statistician(drug, event, stats, feedback=""):
    prompt = f"Drug: {drug}\nEvent: {event}\nFAERS statistics: {json.dumps(stats)}"
    return ask(statistician, prompt + (f"\n\nQC FEEDBACK:\n{feedback}" if feedback else ""))


def run_clinical(drug, event, stat: StatReview, articles, feedback=""):
    prompt = (f"Drug: {drug}\nEvent: {event}\nStatistical summary: {stat.key_numbers}. {stat.interpretation}\n"
              f"Number of PubMed articles found: {len(articles)}\n\nABSTRACTS:\n{format_articles(articles) or 'none'}")
    return ask(clinical_reviewer, prompt + (f"\n\nQC FEEDBACK:\n{feedback}" if feedback else ""))


def drug_safety_analysis(drug: str, event: str, max_iterations: int = 3, threshold: int = 90) -> Dict[str, Any]:
    print(f"━━ Drug safety: {drug} → {event}")
    stats = disproportionality(drug, event)
    audit.log("faers", "disproportionality", **stats)
    print(f"   FAERS: a={stats['table']['a']} ROR={stats['ror']} CI={stats['ror_ci95']} signal={stats['signal']}")
    articles = pubmed_search(f'"{drug}"[Title/Abstract] AND "{event}"[Title/Abstract]', retmax=20)
    audit.log("pubmed", "search", n_articles=len(articles), pmids=[a["pmid"] for a in articles])
    print(f"   PubMed: {len(articles)} articles")

    stat = run_statistician(drug, event, stats)
    clin = run_clinical(drug, event, stat, articles)
    audit.log("statistician", "review", revision=0, **stat.model_dump())
    audit.log("clinical", "review", revision=0, **clin.model_dump())

    history = []
    for iteration in range(1, max_iterations + 1):
        qc = qc_review(stats, stat, clin, articles)
        audit.log("qc", "review", iteration=iteration, **qc)
        history.append({"iteration": iteration, "score": qc["score"], "flagged": qc["targets"],
                        "issues": [i["problem"] for i in qc["issues"]]})
        print(f"   QC iteration {iteration}: score={qc['score']} flagged={qc['targets'] or 'none'}")
        if qc["score"] >= threshold or not qc["targets"] or iteration == max_iterations:
            break
        if "statistician" in qc["targets"]:
            stat = run_statistician(drug, event, stats, feedback_for("statistician", qc["issues"]))
            audit.log("statistician", "revision", revision=iteration, **stat.model_dump())
        if "clinical" in qc["targets"] or "statistician" in qc["targets"]:   # clinical depends on the stats summary
            clin = run_clinical(drug, event, stat, articles, feedback_for("clinical", qc["issues"]))
            audit.log("clinical", "revision", revision=iteration, **clin.model_dump())

    report = regulatory_report(stats, clin)
    audit.log("reporter", "report", **report)
    return {"drug": drug, "event": event, "stats": stats, "n_articles": len(articles), "statistician": stat.model_dump(),
            "clinical": clin.model_dump(), "qc_history": history, "final_qc_score": history[-1]["score"], "report": report}

### Run Part A
`MONTELUKAST → AGGRESSION` is a classic real-world signal (the FDA added a boxed warning for neuropsychiatric
events in 2020) with an incompletely understood mechanism, a good stress test for the feedback loop.
Also try `METFORMIN → LACTIC ACIDOSIS` or `ATORVASTATIN → RHABDOMYOLYSIS`.

In [ ]:
safety = drug_safety_analysis("MONTELUKAST", "AGGRESSION")

print("\nRisk level:", safety["report"]["risk_level"], "| actions:", safety["report"]["actions"])
print("Clinical classification:", safety["clinical"]["classification"], "|", safety["clinical"]["mechanism"])
pd.DataFrame(safety["qc_history"])

# Part B: Drug Discovery (agentic RAG for repurposing)

1. **Retrieve**: search PubMed, embed the abstracts locally, rank them against the question.
2. **Self-grade** the retrieval → confidence in [0, 1]:
   `0.5 × disease coverage (top-k docs that mention the disease) + 0.3 × volume (docs / k) + 0.2 × recency`
3. **Correct**: if confidence < τ, a **query-rewriter agent** proposes new PubMed searches (synonyms, MeSH terms,
   broader phrasing); results are merged and re-graded. This is the "agentic" part of agentic RAG.
4. **Extract** candidates (structured), **guard** citations, **score** transparently.

In [ ]:
TAU = 0.6          # minimum retrieval confidence
TOP_K = 12         # documents passed to the extractor
MAX_REWRITES = 2   # corrective retrieval rounds


class SearchPlan(BaseModel):
    queries: List[str] = Field(description="2-3 alternative PubMed queries (synonyms, MeSH terms, broader wording)")
    reasoning: str


class Candidate(BaseModel):
    drug: str = Field(description="Generic name of an existing drug")
    evidence_tier: Literal["phase-3", "phase-2", "phase-1", "preclinical"]
    model: Literal["human", "animal", "in_vitro"] = Field(description="Best evidence model in the documents")
    outcome: Literal["benefit", "mixed", "no_effect", "harm"]
    mechanism: str = Field(description="Mechanism / target / pathway in a few words")
    pmids: List[str] = Field(description="Supporting PMIDs (digits) from the provided documents")
    confidence: float = Field(description="Confidence in this extraction, 0 to 1")


class Extraction(BaseModel):
    candidates: List[Candidate]


query_rewriter = LLMAgent(llm, output_type=SearchPlan, system_prompt=(
    "You are a biomedical librarian. The previous PubMed searches returned weak evidence. Propose 2-3 better PubMed "
    "queries to find existing drugs studied or repurposed for the disease: use synonyms, MeSH terms "
    "(e.g. 'Drug Repositioning'[MeSH]) and broader wording. Return valid PubMed query syntax."))

extractor = LLMAgent(llm, output_type=Extraction, system_prompt=(
    "You extract drug-repurposing evidence. From the numbered documents ONLY, list existing drugs investigated for "
    "the disease, one entry per drug with its strongest evidence. Never invent drugs, results or PMIDs. Skip drugs "
    "mentioned only in passing. Use outcome 'harm' if the drug worsened the disease."))

In [ ]:
def grade_retrieval(disease: str, docs: List[Dict]) -> float:
    if not docs:
        return 0.0
    words = [w for w in re.findall(r"[a-z0-9]+", disease.lower()) if len(w) > 3] or [disease.lower()]
    covered = sum(all(w in (d["title"] + " " + d["abstract"]).lower() for w in words) for d in docs) / len(docs)
    volume = min(1.0, len(docs) / TOP_K)
    recency = 1.0 if any(d["year"] and date.today().year - d["year"] <= 5 for d in docs) else 0.0
    return round(0.5 * covered + 0.3 * volume + 0.2 * recency, 3)


def rank(question: str, docs: List[Dict], k: int = TOP_K) -> List[Dict]:
    docs = [d for d in docs if d["abstract"]]
    if not docs:
        return []
    sims = embed([f"{d['title']}. {d['abstract'][:2000]}" for d in docs]) @ embed([question])[0]
    for d, s in zip(docs, sims):
        d["similarity"] = round(float(s), 3)
    return sorted(docs, key=lambda d: d["similarity"], reverse=True)[:k]


def agentic_retrieve(disease: str) -> tuple:
    question = f"existing drugs repurposed or repositioned to treat {disease}"
    queries = [f'({disease}) AND ("Drug Repositioning"[MeSH] OR repurpos*[Title/Abstract] OR reposition*[Title/Abstract])']
    pool: Dict[str, Dict] = {}
    trace = []
    for round_ in range(MAX_REWRITES + 1):
        for q in queries:
            for a in pubmed_search(q, retmax=25):
                pool.setdefault(a["pmid"], a)
        top = rank(question, list(pool.values()))
        conf = grade_retrieval(disease, top)
        trace.append({"round": round_, "queries": queries, "pool_size": len(pool), "confidence": conf})
        audit.log("retriever", "grade", round=round_, queries=queries, pool_size=len(pool), confidence=conf)
        print(f"   retrieval round {round_}: {len(pool)} docs, confidence={conf} (tau={TAU})")
        if conf >= TAU or round_ == MAX_REWRITES:
            return top, trace
        plan = ask(query_rewriter, f"Disease: {disease}\nQueries already tried: {[t['queries'] for t in trace]}\n"
                                         f"Confidence: {conf} (needs {TAU}).")
        queries = plan.queries[:3]
        audit.log("query_rewriter", "plan", **plan.model_dump())
        print(f"   ↳ rewriter: {queries}")


EVIDENCE_W = {"phase-3": 1.0, "phase-2": 0.7, "phase-1": 0.4, "preclinical": 0.2}
MODEL_W = {"human": 1.0, "animal": 0.6, "in_vitro": 0.3}
OUTCOME_W = {"benefit": 1.0, "mixed": 0.5, "no_effect": 0.1, "harm": -0.5}


def score(c: Candidate) -> float:
    """0.40 evidence tier + 0.30 quality (model × outcome) + 0.20 volume + 0.10 confidence (+0.1 human benefit ≥ phase 2)."""
    s = (0.40 * EVIDENCE_W[c.evidence_tier] + 0.30 * MODEL_W[c.model] * OUTCOME_W[c.outcome]
         + 0.20 * min(1.0, len(c.pmids) / 3) + 0.10 * max(0.0, min(1.0, c.confidence)))
    if c.model == "human" and c.outcome == "benefit" and c.evidence_tier in ("phase-2", "phase-3"):
        s += 0.1
    return round(s, 3)


def drug_discovery(disease: str) -> Dict[str, Any]:
    print(f"━━ Drug discovery: {disease}")
    docs, trace = agentic_retrieve(disease)
    context = "\n\n".join(f"[{i}] PMID {d['pmid']} ({d['year']}): {d['title']}\n{d['abstract'][:1200]}"
                          for i, d in enumerate(docs, 1))
    extraction = ask(extractor, f"DISEASE: {disease}\n\nDOCUMENTS:\n{context}")
    known = {d["pmid"] for d in docs}
    kept = []
    for c in extraction.candidates:                       # guardrail: only citations we actually retrieved
        c.pmids = sorted({re.sub(r"\D", "", p) for p in c.pmids} & known)
        if c.pmids:
            kept.append({**c.model_dump(), "score": score(c)})
    audit.log("extractor", "candidates", extracted=len(extraction.candidates), kept=len(kept))
    ranked = sorted(kept, key=lambda r: r["score"], reverse=True)
    print(f"   candidates: {len(extraction.candidates)} extracted, {len(kept)} kept after the PMID guardrail")
    return {"disease": disease, "retrieval": trace, "documents": [{k: d[k] for k in ("pmid", "title", "year", "similarity")}
                                                                   for d in docs], "candidates": ranked}

# Part C: Safety-aware discovery
Discovery proposes; safety vets. Each top candidate gets a FAERS snapshot (report volume, share of serious
reports, most frequent reactions), and a writer agent produces a cited brief that weighs efficacy evidence
against the safety profile.

In [ ]:
brief_writer = LLMAgent(llm, output_type=str, system_prompt=(
    "You write concise drug-repurposing briefs in markdown for a pharma R&D team. For each candidate: evidence "
    "level, mechanism, key citations as [PMID 12345], and the FAERS safety profile (note that FAERS counts reflect "
    "reporting volume, not incidence). Finish with a ranked recommendation, the main limitations, and the line "
    "'Research use only, not medical advice.'"))


def safety_aware_discovery(disease: str, top_n: int = 5) -> Dict[str, Any]:
    discovery = drug_discovery(disease)
    top = discovery["candidates"][:top_n]
    for c in top:
        c["safety"] = safety_snapshot(c["drug"])
        audit.log("safety_screen", "faers_snapshot", **c["safety"])
    table = "\n".join(f"{i}. {c['drug']} score={c['score']} tier={c['evidence_tier']} model={c['model']} "
                      f"outcome={c['outcome']} mechanism={c['mechanism']} PMIDs={c['pmids']} safety={c['safety']}"
                      for i, c in enumerate(top, 1))
    brief = ask(brief_writer, f"Disease: {disease}\nRetrieval confidence: {discovery['retrieval'][-1]['confidence']}\n\n"
                                    f"CANDIDATES:\n{table or 'none found'}")
    audit.log("brief_writer", "brief", chars=len(brief))
    return {**discovery, "top": top, "brief": brief}

### Run Parts B + C

In [ ]:
from IPython.display import Markdown

discovery = safety_aware_discovery("idiopathic pulmonary fibrosis")
pd.DataFrame(discovery["top"], columns=["drug", "score", "evidence_tier", "model", "outcome", "mechanism", "pmids", "safety"])

In [ ]:
Markdown(discovery["brief"])

## 4. Save the run
Everything needed to reproduce or review the run: results JSON, the markdown brief and the audit trail.

In [ ]:
with open(os.path.join(RUN_DIR, "results.json"), "w") as f:
    json.dump({"run_id": RUN_ID, "model": MODEL_NAME, "framework": "langchain",
               "safety": safety, "discovery": discovery}, f, indent=2, default=str)
with open(os.path.join(RUN_DIR, "brief.md"), "w") as f:
    f.write(discovery["brief"])
print("Saved to", RUN_DIR, "→", sorted(os.listdir(RUN_DIR)))
pd.read_json(os.path.join(RUN_DIR, "audit.jsonl"), lines=True)[["ts", "agent", "action"]]